# Step 03. A* 알고리즘

> **출처**: PythonRobotics (https://github.com/AtsushiSakai/PythonRobotics)  
> Original author: Atsushi Sakai (@Atsushi_twi) / License: MIT  
>  
> **교육 주관**: Hanyang University, Automotive Intelligence Laboratory (AILAB)

Step 02 Dijkstra는 **방사형 전방위 탐색**이라 목표와 무관한 노드도 많이 봤습니다.  
A\*는 **Heuristic** `h(n)`을 더해 목표 방향으로 집중 탐색합니다.

> Step 02처럼 작은 조각으로 나눠 이해하고, 마지막에 **Dijkstra와 나란히 비교**합니다.

## 학습 목표

1. `f = g + h` 우선순위 이해
2. Admissible heuristic (과대추정 없음 → 최적성)
3. Heuristic 필드가 탐색을 유도하는 모습
4. Dijkstra vs A\* 탐색 과정 **동시 재생** 비교

In [ ]:
%matplotlib inline
import sys, os, math, heapq, time
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML

from utils.grid_map import make_simple_map, build_obstacle_map
from utils.visualization import plot_grid_map, plot_search_result
from utils.grid_search import (calc_grid_index, calc_grid_position, node_key,
                               get_motion_model, is_valid, reconstruct_path, dijkstra_search)
from utils.search_viz import animate_search, animate_compare

ox, oy, sx, sy, gx, gy = make_simple_map()
RESOLUTION, ROBOT_RADIUS = 2.0, 1.0
min_x, max_x = round(min(ox)), round(max(ox))
min_y, max_y = round(min(oy)), round(max(oy))
x_width = round((max_x-min_x)/RESOLUTION); y_width = round((max_y-min_y)/RESOLUTION)

print('환경 로드 완료 (Step 02와 동일한 맵 -> 탐색 효율 직접 비교)')
fig, ax = plt.subplots(figsize=(6, 6))
plot_grid_map(ox, oy, sx, sy, gx, gy, ax=ax); ax.set_title('Search Environment')
plt.tight_layout(); plt.show()


## 1. Heuristic 함수

### 이론 배경

`h(n)` = 현재 → 목표 **예상** 비용. **Admissible**(실제 비용을 절대 과대추정 안 함)이면 A\*가 최적해를 보장합니다.  
Euclidean은 admissible, Manhattan은 대각선 이동에서 과대추정이라 부적합.

In [ ]:
def heuristic(node, goal):
    # Euclidean distance — 허용 가능(admissible) heuristic (실제 비용 과대추정 안 함)
    return math.hypot(node.x - goal.x, node.y - goal.y)

class _P:                      # 비교 출력용 임시 점
    def __init__(s, x, y): s.x, s.y = x, y

a, b = _P(0, 0), _P(3, 4)      # 실제 거리 = 5.0
print('Heuristic 비교 ((0,0) -> (3,4), 실제거리 5.0)')
print(f'  Euclidean : {math.hypot(3,4):.2f}  (admissible, 최적성 보장)')
print(f'  Manhattan : {3+4:.2f}  (대각선 이동 시 과대추정 -> 부적합)')
print(f'  Zero      : 0.00  (= Dijkstra, 비효율)')


## 2. Heuristic 필드

### 이론 배경

격자 전체의 `h(n)` 을 색으로 그리면 목표 쪽이 어두워지는 **기울기**가 보입니다.  
A\*는 `g(n)`(실제 비용)에 이 `h(n)`을 더해, 목표 방향으로 탐색을 기울입니다.

In [ ]:
# 격자 전체에서 목표까지의 h(n) 값 (어두울수록 목표에 가까움)
gxi = (gx - min_x)/RESOLUTION; gyi = (gy - min_y)/RESOLUTION
H = np.array([[math.hypot(ix-gxi, iy-gyi) for ix in range(x_width)] for iy in range(y_width)])

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(H, origin='lower', extent=[min_x,max_x,min_y,max_y], cmap='viridis_r', alpha=0.9)
ax.plot(ox, oy, '.k', ms=2, alpha=0.4)
ax.plot(sx, sy, 'og', ms=12, label='Start'); ax.plot(gx, gy, 'xr', ms=12, mew=2.5, label='Goal (h=0)')
plt.colorbar(im, ax=ax, label='estimated cost to goal', fraction=0.046)
ax.set_aspect('equal'); ax.legend(loc='upper left', fontsize=8)
ax.set_title('Heuristic Field h(n)  (A* follows this gradient toward goal)')
plt.tight_layout(); plt.show()
print('A*는 이 기울기를 따라 목표 방향으로 탐색을 집중한다')


## 3. Node 클래스

### 이론 배경

Step 02와 동일한 Node (격자 인덱스 + 누적 비용 g + 부모 키).  
좌표변환·is_valid·역추적은 `utils.grid_search`에서 그대로 재사용합니다.

In [ ]:
class Node:
    def __init__(self, x, y, cost, parent_index):
        self.x = x; self.y = y
        self.cost = cost                  # g(n): 출발 -> 현재 실제 비용
        self.parent_index = parent_index
    def __lt__(self, other):
        return self.cost < other.cost

print('Node 정의 완료 (좌표변환/is_valid/역추적은 utils.grid_search 재사용)')


## 4. Dijkstra와의 단 한 줄 차이

### 이론 배경

우선순위 큐에 넣는 값만 `g` → `g + h(n)` 으로 바뀝니다. 그 외 구조는 동일합니다.

In [ ]:
# Dijkstra와 A*의 유일한 차이: 우선순위 큐에 넣는 값
print('Dijkstra:  heapq.heappush(pq, (g,        node))   # g(n)')
print('A*      :  heapq.heappush(pq, (g + h(n), node))   # f = g + h')
print()
print('나머지(Open/Closed Set, 이웃 확장, 역추적)는 Step 02와 완전히 동일')


## 5. astar() 조립

### 이론 배경

Step 02의 `dijkstra()` 골격에 `heappush` 한 줄을 `f = g + h`로 바꿔 조립합니다.  
`explore_order`도 반환해 동일하게 애니메이션에 사용합니다.

In [ ]:
def astar(sx, sy, gx, gy, ox, oy, resolution, robot_radius):
    obstacle_map, min_x, min_y, max_x, max_y, x_width, y_width = \
        build_obstacle_map(ox, oy, resolution, robot_radius)
    start = Node(calc_grid_index(sx, min_x, resolution), calc_grid_index(sy, min_y, resolution), 0.0, -1)
    goal  = Node(calc_grid_index(gx, min_x, resolution), calc_grid_index(gy, min_y, resolution), 0.0, -1)

    open_set = {node_key(start, x_width, min_x, min_y): start}
    closed_set = {}
    pq = [(heuristic(start, goal), start)]      # f = g(0) + h
    explore_order = []

    while pq:
        _, current = heapq.heappop(pq)
        cid = node_key(current, x_width, min_x, min_y)
        if cid not in open_set: continue
        open_set.pop(cid); closed_set[cid] = current
        explore_order.append((calc_grid_position(current.x, min_x, resolution),
                              calc_grid_position(current.y, min_y, resolution), current.cost))
        if current.x == goal.x and current.y == goal.y:
            goal.parent_index = current.parent_index; goal.cost = current.cost; break
        for dx, dy, cost in get_motion_model():
            nb = Node(current.x+dx, current.y+dy, current.cost+cost, cid)
            nk = node_key(nb, x_width, min_x, min_y)
            if not is_valid(nb, obstacle_map, min_x, min_y, max_x, max_y, resolution): continue
            if nk in closed_set: continue
            if nk not in open_set or open_set[nk].cost > nb.cost:
                open_set[nk] = nb
                heapq.heappush(pq, (nb.cost + heuristic(nb, goal), nb))   # f = g + h
    path_x, path_y = reconstruct_path(goal, closed_set, min_x, min_y, resolution)
    closed_x = [calc_grid_position(n.x, min_x, resolution) for n in closed_set.values()]
    closed_y = [calc_grid_position(n.y, min_y, resolution) for n in closed_set.values()]
    return path_x, path_y, closed_x, closed_y, explore_order

print('astar() 정의 완료 — Step 02 dijkstra()에서 push 한 줄만 f=g+h 로 변경')


## 6. 실행 및 Dijkstra 비교

### 이론 배경

같은 맵에서 두 알고리즘의 탐색 노드 수를 비교합니다.  
경로 길이는 동일(둘 다 최적), 탐색 노드 수는 A\*가 적을수록 heuristic이 효과적.

In [ ]:
# A* 와 Dijkstra 를 같은 환경에서 실행해 탐색 노드 수 비교
t0 = time.time()
path_x, path_y, closed_x, closed_y, explore_order = astar(sx,sy,gx,gy,ox,oy,RESOLUTION,ROBOT_RADIUS)
astar_t = (time.time()-t0)*1000

t0 = time.time()
d_px, d_py, d_cx, d_cy, d_order = dijkstra_search(sx,sy,gx,gy,ox,oy,RESOLUTION,ROBOT_RADIUS, Node)
dijk_t = (time.time()-t0)*1000

reduction = (1 - len(closed_x)/len(d_cx))*100
print(f'Dijkstra : {len(d_cx):,} 노드, {dijk_t:.0f} ms')
print(f'A*       : {len(closed_x):,} 노드, {astar_t:.0f} ms')
print(f'-> 탐색 노드 {reduction:.0f}% 감소 (경로 길이는 동일)')

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(['Dijkstra','A*'], [len(d_cx), len(closed_x)], color=['#4C72B0','#C44E52'], alpha=0.9)
for i,v in enumerate([len(d_cx), len(closed_x)]): ax.text(i, v, f'{v:,}', ha='center', va='bottom')
ax.set_ylabel('explored nodes'); ax.set_title('Explored Node Count')
plt.tight_layout(); plt.show()


## 7. 탐색 과정 동시 재생

### 이론 배경

왼쪽 Dijkstra(방사형) vs 오른쪽 A\*(목표 방향 집중)를 **같은 프레임에 나란히** 재생합니다.  
A\*의 탐색 영역이 목표 쪽으로 좁게 뻗는 것을 직접 비교하세요.

In [ ]:
# 두 알고리즘의 탐색 과정을 좌우로 동시 재생 (utils.search_viz.animate_compare)
HTML(animate_compare(d_order, d_px, d_py, explore_order, path_x, path_y,
                     ox, oy, sx, sy, gx, gy, label_a='Dijkstra', label_b='A*').to_jshtml())


## 8. 결과 요약

### A\* 특징

| 항목 | Dijkstra | A\* |
|------|----------|----|
| 우선순위 | g | g + h |
| 탐색 | 방사형 | 목표 집중 |
| 최적성 | 보장 | admissible h면 보장 |

→ Step 04 Hybrid A\*: 연속 상태 (x,y,yaw) + 차량 기구학

In [ ]:
fig = plot_search_result(closed_x, closed_y, path_x, path_y, ox, oy, sx, sy, gx, gy,
                         title=f'A* ({len(closed_x):,} nodes)')
plt.tight_layout(); plt.show()
print('A*: admissible heuristic이면 Dijkstra와 같은 최적 경로를 더 적은 탐색으로')
print('다음 Step 04 Hybrid A*: 격자 (ix,iy) -> 연속 (x,y,yaw), Bicycle Model로 차량 주행 가능 경로')
